### ЗАДАЧА: Панель претензий к поставщикам по недопоставке по паттерну `MVC`

Команда procurement operations разбирает кейсы по недопоставке товара от поставщиков.
После приемки поставки система сравнивает ожидаемое количество и фактически полученный товар.
Если есть расхождение, создается кейс, по которому нужно провести расследование, запросить документы,
согласовать компенсацию от поставщика и закрыть кейс.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `supplier` — поставщик;
- `shipment_id` — идентификатор поставки;
- `sku` — товар;
- `expected_qty` — ожидаемое количество;
- `received_qty` — фактически принятое количество;
- `unit_cost` — себестоимость единицы товара;
- `claim_qty` — количество товара, которое считается недопоставленным;
- `claim_amount` — сумма претензии к поставщику;
- `approved_compensation` — согласованная компенсация;
- `remaining_loss` — остаток убытка после компенсации;
- `status` — текущий статус кейса;
- `manager` — сотрудник, который ведет кейс;
- `documents_verified` — подтверждены ли документы;
- `decision` — итоговое решение.

## Формулы

При создании кейса и после любого изменения компенсации нужно правильно считать:
- `claim_qty = expected_qty - received_qty`
- если `claim_qty < 0`, нужно выбрасывать ошибку;
- `claim_amount = claim_qty * unit_cost`
- `remaining_loss = claim_amount - approved_compensation`
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

- `new`
- `investigating`
- `documents_requested`
- `documents_verified`
- `ready_for_resolution`
- `fully_compensated`
- `partial_compensated`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `manager` несуществующему кейсу;
- финальные кейсы (`fully_compensated`, `partial_compensated`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `manager`;
- запросить документы можно только из `investigating`;
- подтвердить документы можно только из `documents_requested`;
- при подтверждении документов поле `documents_verified` должно стать `True`, а статус — `documents_verified`;
- установить `approved_compensation` можно только из `investigating` или `documents_verified`;
- `approved_compensation` не может быть меньше `0`;
- `approved_compensation` не может быть больше `claim_amount`;
- после изменения `approved_compensation` нужно пересчитать `remaining_loss`;
- перевод в `ready_for_resolution` возможен только из `investigating` или `documents_verified`;
- перевод в `ready_for_resolution` невозможен, если `approved_compensation == 0` и при этом `documents_verified == False`;
- завершить кейс как `fully_compensated` можно только из `ready_for_resolution`, если `approved_compensation == claim_amount`;
- завершить кейс как `partial_compensated` можно только из `ready_for_resolution`, если `0 < approved_compensation < claim_amount`;
- завершить кейс как `rejected` можно только из `ready_for_resolution`, если `approved_compensation == 0`;
- эскалировать кейс можно только из `investigating`, `documents_verified` или `ready_for_resolution`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать менеджера;
- начинать расследование;
- запрашивать документы;
- подтверждать документы;
- устанавливать `approved_compensation`;
- переводить кейс в `ready_for_resolution`;
- завершать кейс как `fully_compensated`;
- завершать кейс как `partial_compensated`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, вывести отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- оборачивать операции в `try/except` с `ValueError`;
- передавать результат во view;
- обработать все действия из `actions`.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | supplier | shipment_id | sku | expected_qty | received_qty | claim_qty | claim_amount | approved_compensation | remaining_loss | status | manager | documents_verified | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_claim_amount` — общая сумма претензий;
- `total_approved_compensation` — общая согласованная компенсация;
- `total_remaining_loss` — общий остаток убытка;
- `verified_docs_cases` — количество кейсов, где документы подтверждены;
- `fully_compensated_amount` — сумма компенсаций по кейсам со статусом `fully_compensated`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [ ]:
from dataclasses import dataclass

initial_cases = [
    ("SC-100", "alpha-supply", "SHIP-7001", "SKU-100", 120, 102, 35.0),
    ("SC-101", "beta-distribution", "SHIP-7002", "SKU-200", 80, 80, 50.0),
]

actions = [
    ("show",),
    ("investigate", "SC-100"),
    ("assign", "SC-100", "Olga"),
    ("investigate", "SC-100"),
    ("docs_request", "SC-100"),
    ("docs_verify", "SC-100"),
    ("set_compensation", "SC-100", 500.0),
    ("ready", "SC-100"),
    ("partial", "SC-100", "supplier_accepted_partial_claim"),
    ("create", "SC-102", "gamma-warehouses", "SHIP-7003", "SKU-300", 60, 40, 28.0),
    ("assign", "SC-102", "Max"),
    ("investigate", "SC-102"),
    ("set_compensation", "SC-102", 560.0),
    ("ready", "SC-102"),
    ("full", "SC-102", "full_compensation_approved"),
    ("create", "SC-103", "delta-trade", "SHIP-7004", "SKU-400", 45, 30, 42.0),
    ("assign", "SC-103", "Ina"),
    ("investigate", "SC-103"),
    ("escalate", "SC-103", "supplier_disputes_shortage"),
    ("show",),
]
@dataclass
class Case:
    case_id: str
    supplier: str
    shipment_id: str
    sku: str
    expected_qty: int
    received_qty: int
    unit_cost: int
    claim_qty: int = 0.0
    claim_amount: int = 0.0
    approved_compensation: int = 0.0
    remaining_loss: int = 0.0
    status: str = 'new'
    manager: str = None
    documents_verified: bool = None
    decision: str = None

class Model:
    final_statuses = {'fully_compensated', 'partial_compensated', 'rejected', 'escalated'}
    def __init__(self):
        self.cases = {}
    
    def _calc_claim_qty(self, expected_qty, received_qty):
        claim_qty = round(expected_qty - received_qty,2)
        if claim_qty < 0:
            raise ValueError('invalid claim_qty')
        return claim_qty

    def _calc_claim_amount(self, claim_qty, unit_cost):
        return round(claim_qty * unit_cost, 2)
    
    def _calc_remaining_loss(self, claim_amount, approved_compensation):
        return round(claim_amount - approved_compensation, 2)
    
    def add_case(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        if case_id in self.cases:
            raise ValueError('Case already exists')
        claim_qty = self._calc_claim_qty(expected_qty, received_qty)
        claim_amount = self._calc_claim_amount(claim_qty, unit_cost)
        remaining_loss = self._calc_remaining_loss(claim_amount, 0.0)
        self.cases[case_id] = Case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost,claim_qty=claim_qty, claim_amount=claim_amount, remaining_loss=remaining_loss)
    
    def _get_case(self, case_id) -> Case:
        if case_id not in self.cases:
            raise ValueError('Case not found')
        return self.cases[case_id]
    def assign_manager(self, case_id, manager):
        case = self._get_case(case_id)
        if case.status in self.final_statuses:
            raise ValueError('Cannot change case with final status')
        case.manager = manager

    def start_investigation(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'new' or case.manager is None:
            raise ValueError('Manager is required')
        case.status = 'investigating'
    def request_document(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'investigating':
            raise ValueError('invalid status for requesting document')
        case.status = 'documents_requested'
    
    def confirm_document(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'documents_requested':
            raise ValueError('invalid status for confirming document')
        case.documents_verified = True
        case.status = 'documents_verified'

    def set_approved_compensation(self, case_id, approved_compensation):
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'documents_verified'}:
            raise ValueError('invalid status for setting approved compensation')
        if approved_compensation < 0:
            raise ValueError('approved compensation must be >= 0')
        if approved_compensation > case.claim_amount:
            raise ValueError('invalid approved compensation')
        case.approved_compensation = approved_compensation
        case.remaining_loss = self._calc_remaining_loss(case.claim_amount, approved_compensation)

    def mark_ready_for_resolution(self, case_id):
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'documents_verified'}:
            raise ValueError('invalid status for marking case ready for resolution')
        if case.approved_compensation == 0 or case.documents_verified is False:
            raise ValueError('Cannot mark ready for resolution')
        case.status = 'ready_for_resolution'

    def finish_as_fully_compensated(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'ready_for_resolution' or case.approved_compensation != case.claim_amount:
            raise ValueError('Cannot finish case as fully compensated')
        case.status = 'fully_compensated'
        case.decision = 'fully_compensated'
    
    def finish_as_partial_compensated(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'ready_for_resolution' or not 0 < case.approved_compensation < case.claim_amount:
            raise ValueError('Cannot finish case as partial compensated')
        case.decision = 'partial_compensated'
        case.status = 'partial_compensated'
    
    def reject_case(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'ready_for_resolution' or case.approved_compensation != 0:
            raise ValueError('Cannot reject a case')
        case.status = 'rejected'
        case.decision = 'rejected'
    
    def escalate_case(self, case_id):
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'documents_verified', 'ready_for_resolution'}:
            raise ValueError('invalid status for escalating a case')
        case.status = 'escalated'
        case.decision = 'escalated'
    
    def summary(self):
        res = {
            'total_claim_amount': 0,
            'total_approved_compensation': 0,
            'total_remaining_loss': 0,
            'verified_docs_cases': 0,
            'fully_compensated_amount': 0,
        
        }
        for case in self.cases.values():
            res[case.status] = res.get(case.status, 0) + 1
            res['total_claim_amount'] += case.claim_amount
            res['total_approved_compensation'] += case.approved_compensation
            res['total_remaining_loss'] += case.remaining_loss
            res['verified_docs_cases'] += 1 if case.documents_verified == True else 0
            res['fully_compensated_amount'] += case.approved_compensation if case.status == 'fully_compensated' else 0
        return res
    def list_cases(self):
        arr = []
        for case in self.cases.values():
            arr.append(case)
        return arr

class View:
    def show_cases(self, cases):
        print('Cases: ')
        if not cases:
            print('The cases list is empty!')
            return
        for case in cases:
            print(f'{case.case_id} | {case.supplier} | {case.shipment_id} | {case.sku} | {case.expected_qty} | {case.received_qty} | {case.claim_qty} | {case.claim_amount} | {case.approved_compensation} | {case.remaining_loss} | {case.status} | {case.manager} | {case.documents_verified} | {case.decision}')
    
    def show_summary(self, summary):
        print(f'Summary: {summary}')
    
    def render_success(self, msg):
        print(f'SUCCESS: {msg}')
    
    def render_error(self, msg):
        print(f'ERROR: {msg}')

class Controller:
    def __init__(self, model: Model, view: View):
        self.model = model
        self.view = view
    
    def create_case(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        try:
            self.model.add_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
            self.view.render_success(f"Case {case_id} added")
        except ValueError as e:
            self.view.render_error(e)
    def assign_manager(self, case_id, manager):
        try:
            self.model.assign_manager(case_id, manager)
            self.view.render_success(f'Manager to case {case_id} assigned')
        except ValueError as e:
            self.view.render_error(e)
    
    def start_investigation(self, case_id):
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f'Investigation for case {case_id} started')
        except ValueError as e:
            self.view.render_error(e)
    
    def request_document(self, case_id):
        try:
            self.model.request_document(case_id)
            self.view.render_success(f'Document for case {case_id} requested')
        except ValueError as e:
            self.view.render_error(e)
    def confirm_document(self, case_id):
        try:
            self.model.confirm_document(case_id)
            self.view.render_success(f'Document for case {case_id} confirmed')
        except ValueError as e:
            self.view.render_error(e)
    
    def set_approved_compensation(self, case_id, comp):
        try:
            self.model.set_approved_compensation(case_id, comp)
            self.view.render_success(f'Approved compensation for case {case_id} set')
        except ValueError as e:
            self.view.render_error(e)
    
    def mark_ready_for_resolution(self, case_id):
        try:
            self.model.mark_ready_for_resolution(case_id)
            self.view.render_success(f'Case {case_id} marked ready for resolution')
        except ValueError as e:
            self.view.render_error(e)

    def finish_as_fully_compensated(self, case_id, msg):
        try:
            self.model.finish_as_fully_compensated(case_id)
            self.view.render_success(f'Case {case_id} finished as fully compensated: {msg}')
        except ValueError as e:
            self.view.render_error(e)
    
    def finish_as_partial_compensated(self, case_id, msg):
        try:
            self.model.finish_as_partial_compensated(case_id)
            self.view.render_success(f'Case {case_id} finished as partial compensated: {msg}')
        except ValueError as e:
            self.view.render_error(e)
    
    def reject_case(self, case_id):
        try:
            self.model.reject_case(case_id)
            self.view.render_success(f'Case {case_id} rejected')
        except ValueError as e:
            self.view.render_error(e)
    
    def escalate_case(self, case_id, msg):
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f'Case {case_id} escalated: {msg}')
        except ValueError as e:
            self.view.render_error(e)
    
    def show_cases(self):
        self.view.show_cases(self.model.list_cases())
        self.view.show_summary(self.model.summary())

model = Model()
view = View()
controller = Controller(model=model, view=view)
for case in initial_cases:
    model.add_case(case[0], case[1], case[2], case[3], case[4], case[5], case[6])

for act in actions:
    if act[0] == 'show':
        controller.show_cases()
    if act[0] == 'investigate':
        controller.start_investigation(act[1])
    if act[0] == 'docs_request':
        controller.request_document(act[1])
    if act[0] == 'docs_verify':
        controller.confirm_document(act[1])
    if act[0] == 'set_compensation':
        controller.set_approved_compensation(act[1], act[2])
    if act[0] == 'ready':
        controller.mark_ready_for_resolution(act[1])
    if act[0] == 'partial':
        controller.finish_as_partial_compensated(act[1], act[2])
    if act[0] == 'create':
        controller.create_case(act[1], act[2], act[3], act[4], act[5], act[6], act[7])
    if act[0] == 'assign':
        controller.assign_manager(act[1], act[2])
    if act[0] == 'full':
        controller.finish_as_fully_compensated(act[1], act[2])
    if act[0] == 'escalate':
        controller.escalate_case(act[1], act[2])
    
print('Финальное состояние: ')
controller.show_cases()


Cases: 
SC-100 | alpha-supply | SHIP-7001 | SKU-100 | 120 | 102 | 0.0 | 0.0 | 0.0 | 0.0 | new | None | None | None
SC-101 | beta-distribution | SHIP-7002 | SKU-200 | 80 | 80 | 0.0 | 0.0 | 0.0 | 0.0 | new | None | None | None
Summary: {'total_claim_amount': 0.0, 'total_approved_compensation': 0.0, 'total_remaining_loss': 0.0, 'verified_docs_cases': 0, 'fully_compensated_amount': 0, 'new': 2}
ERROR: Manager is required
SUCCESS: Manager to case SC-100 assigned
SUCCESS: Investigation for case SC-100 started
SUCCESS: Document for case SC-100 requested
SUCCESS: Document for case SC-100 confirmed
ERROR: invalid approved compensation
ERROR: Cannot mark ready for resolution
ERROR: Cannot finish case as partial compensated
SUCCESS: Case SC-102 added
SUCCESS: Manager to case SC-102 assigned
SUCCESS: Investigation for case SC-102 started
ERROR: invalid approved compensation
ERROR: Cannot mark ready for resolution
ERROR: Cannot finish case as fully compensated
SUCCESS: Case SC-103 added
SUCCESS: Ma